# Instalación de librerías y comprobación de entorno


In [ ]:
%%capture
# Desinstalar versiones conflictivas si las hay
!pip uninstall -y torch torchvision torchaudio bitsandbytes transformers peft trl

# PyTorch con soporte CUDA 12.1 (compatible con Colab T4 y A10G)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Stack de fine-tuning
!pip install \
    transformers==4.46.3 \
    tokenizers \
    accelerate \
    peft==0.13.2 \
    trl==0.12.1 \
    bitsandbytes==0.44.1 \
    datasets

# Métricas y tracking
!pip install scikit-learn mlflow python-dotenv pandas

In [ ]:
import os
import json
import torch
import numpy as np
import pandas as pd
from pathlib import Path
import sys

# En Colab: subir functions.py a /content/ junto con grading_dataset.jsonl
sys.path.insert(0, "/content")
from functions import (
    LABEL_RELEVANTE, LABEL_NO_RELEVANTE, LABELS, GRADING_SYSTEM_PROMPT,
    check_gpu, load_grading_dataset, split_dataset, show_split_stats,
    build_grading_messages, format_training_prompt, examples_to_hf_dataset,
    load_tokenizer, load_model_4bit, build_peft_model, get_training_args,
    run_training, save_adapter, predict_relevance, evaluate_model,
    print_comparison, log_experiment,
)

check_gpu()

In [ ]:
# ============================================================
# MONTAR GOOGLE DRIVE (para guardar el modelo de forma persistente)
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

# ============================================================
# CONFIGURACIÓN DE RUTAS Y CONSTANTES
# ============================================================

# Dataset — copiar grading_dataset.jsonl al root de Colab antes de ejecutar
DATASET_PATH = Path("/content/grading_dataset.jsonl")

LABEL_RELEVANTE    = "relevante"
LABEL_NO_RELEVANTE = "no relevante"
LABELS = [LABEL_RELEVANTE, LABEL_NO_RELEVANTE]

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

# Salida en Google Drive — persiste entre sesiones de Colab
DRIVE_BASE   = Path("/content/drive/MyDrive/normabot")
OUTPUT_DIR   = str(DRIVE_BASE / "qwen-grader-lora")
ADAPTER_PATH = str(DRIVE_BASE / "qwen-grader-lora" / "adapter_final")
MERGED_PATH  = str(DRIVE_BASE / "qwen-grader-merged")   # solo si se hace merge

MAX_SEQ_LEN = 512

print("Configuración:")
print(f"  Modelo base:    {MODEL_ID}")
print(f"  Dataset:        {DATASET_PATH}  (existe: {DATASET_PATH.exists()})")
print(f"  Output dir:     {OUTPUT_DIR}")
print(f"  Adapter path:   {ADAPTER_PATH}")
print(f"  Max seq len:    {MAX_SEQ_LEN}")

In [ ]:
# Rellenar con los valores obtenidos en NB2 (entrenamiento) y NB3 (pruebas)
baseline_acc  = 0.0   # RELLENAR
baseline_f1   = 0.0   # RELLENAR
finetuned_acc = 0.0   # RELLENAR
finetuned_f1  = 0.0   # RELLENAR
train_loss    = 0.0   # RELLENAR (de NB2 — train_result.training_loss)
train_size    = 0     # RELLENAR
val_size      = 0     # RELLENAR
test_size     = 0     # RELLENAR

print(f'baseline_f1={baseline_f1:.4f}  finetuned_f1={finetuned_f1:.4f}  mejora={finetuned_f1 - baseline_f1:+.4f}')

# Seguimiento con MLflow

Registramos los parámetros y métricas en el servidor MLflow del proyecto
(configurado via `MLFLOW_TRACKING_URI` en `.env`), siguiendo el mismo patrón
que el clasificador de riesgo (`src/classifier/`).


In [ ]:
try:
    log_experiment(
        baseline_acc=baseline_acc,
        baseline_f1=baseline_f1,
        finetuned_acc=finetuned_acc,
        finetuned_f1=finetuned_f1,
        train_loss=train_loss,
        train_size=train_size,
        val_size=val_size,
        test_size=test_size,
        max_seq_len=MAX_SEQ_LEN,
    )
except Exception as e:
    print(f"MLflow no disponible: {e}")
    print(f"  Accuracy: {finetuned_acc:.4f} | F1-macro: {finetuned_f1:.4f}")

# Conclusiones

## Resultados

| Modelo | Accuracy | F1-macro |
|--------|:--------:|:--------:|
| Qwen 2.5 3B-Instruct (baseline, prompting) | — | — |
| Qwen 2.5 3B-Instruct + QLoRA (fine-tuned) | — | — |

*Rellenar tras ejecutar el notebook completo.*

## Dataset utilizado

- **634 ejemplos** · 117 queries sobre EU AI Act, RGPD y AESIA
- Split: 70% train / 15% val / 15% test · balanceo: 44.6% relevantes / 55.4% no relevantes
- Negativos: hard negatives del mismo dominio legal + easy negatives de leyes ajenas
- Generado con `data/generate_grading_dataset.py`

## Limitaciones

- **Tamaño del dataset**: 634 ejemplos es suficiente para un primer experimento pero modesto.
  Ampliar con más queries o con datos reales del retriever ChromaDB mejorará la generalización.
- **Datos sintéticos**: los fragmentos documentales del dataset se generaron a partir de las fuentes
  reales (EU AI Act, RGPD, AESIA) pero no son los chunks exactos de ChromaDB.
  Idealmente el dataset final usaría los chunks reales que devuelve el retriever en producción.
- **Integración con Ollama**: el adaptador LoRA requiere cargar el modelo base en memoria.
  Para producción, hacer merge y convertir a GGUF.

## Próximos pasos

1. **Ejecutar en Colab T4** y rellenar la tabla de resultados
2. **Ampliar el dataset** con chunks reales del retriever ChromaDB
3. **Integrar** en `src/rag/main.py` con la función `grade_with_finetuned()`
4. *(Opcional)* Merge + conversión a GGUF + push a Ollama para producción

## Impacto en el pipeline

Un grader especializado reduce el ruido en el contexto de generación:
- Menos documentos irrelevantes → respuestas más precisas y menos alucinaciones en citas legales
- Menor coste de tokens en la llamada a Bedrock Nova Lite (contexto más corto)

---
_Informe preliminar generado por IA. Consulte profesional jurídico._
